In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.windowing.make_windows import build_dataset

plt.style.use('seaborn-v0_8-whitegrid')

csv_path = Path('../data/features.csv')
df = pd.read_csv(csv_path)
print('shape:', df.shape)
print('subjects:', sorted(df['subject_id'].unique().tolist()))
print('class_counts:')
print(df['class_label'].value_counts().sort_index().to_string())
print('n_missing:', int(df.isna().sum().sum()))
print('\nsubject-level class counts:\n', df.groupby(['subject_id', 'class_label']).size().unstack(fill_value=0).to_string())

feature_cols = ['pitch', 'yaw', 'gaze_ratio']
print('\nfeature means:\n', df[feature_cols].mean().round(4).to_string())
print('\nfeature stds:\n', df[feature_cols].std().round(4).to_string())
print('\nfeature mins/maxs:\n', df[feature_cols].agg(['min', 'max']).round(4).to_string())
print('\nclass feature means:\n', df.groupby('class_label')[feature_cols].mean().round(4).to_string())
print('\nclass feature stds:\n', df.groupby('class_label')[feature_cols].std().round(4).to_string())

# Correlation matrix and class histograms
print('\nfeature correlation matrix:\n', df[feature_cols].corr().round(4).to_string())

for col in feature_cols:
    plt.figure(figsize=(8, 4))
    sns.histplot(data=df, x=col, hue='class_label', bins=35, element='step', common_norm=False)
    plt.title(f'{col} distribution by class')
    plt.tight_layout()
    plt.show()

# Window-level summary for the actual CNN input
windows = build_dataset('data/features.csv', window_size=60, stride=10)
window_counts = pd.Series([label for _, label, _ in windows]).value_counts().sort_index()
print('\nwindow class counts:\n', window_counts.to_string())
print('total windows:', len(windows))


In [ ]:
# Five-second reading-detector EDA
from src.features.motion import MOTION_FEATURE_NAMES, add_motion_features
from src.windowing.make_windows import build_reading_dataset

reading_feature_names = feature_cols + MOTION_FEATURE_NAMES
reading_windows = build_reading_dataset(str(csv_path), window_size=75, stride=8, sampling_fps=15)
reading_window_df = pd.DataFrame({
    'class_label': [label for _, label, _ in reading_windows],
    'subject_id': [subject for _, _, subject in reading_windows],
})

print('five-second window shape:', reading_windows[0][0].shape)
print('five-second window count:', len(reading_windows))
print('binary window counts:')
print(reading_window_df['class_label'].value_counts().sort_index().rename({0: 'not_reading', 1: 'reading'}).to_string())

# Plot 1: binary five-second window balance
plt.figure(figsize=(7, 4))
sns.countplot(data=reading_window_df, x='class_label', order=[0, 1])
plt.xticks([0, 1], ['Not reading', 'Reading'])
plt.title('Five-second CNN window balance')
plt.xlabel('Target')
plt.ylabel('Number of windows')
plt.tight_layout()
plt.show()

# Build frame-level motion features separately within each recording group.
augmented_groups = []
for (subject_id, class_label), group in df.groupby(['subject_id', 'class_label'], sort=True):
    group = group.sort_values('frame_number').copy()
    augmented = add_motion_features(
        group[feature_cols].to_numpy(dtype='float32'),
        sampling_fps=15,
    )
    augmented_frame = pd.DataFrame(augmented, columns=reading_feature_names)
    augmented_frame['subject_id'] = subject_id
    augmented_frame['class_label'] = int(class_label == 2)
    augmented_groups.append(augmented_frame)

motion_df = pd.concat(augmented_groups, ignore_index=True)

# Plot 2: temporal motion-feature distributions by reading target.
plot_motion_cols = ['yaw_velocity', 'gaze_velocity', 'horizontal_motion']
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for axis, column in zip(axes, plot_motion_cols):
    sns.histplot(
        data=motion_df,
        x=column,
        hue='class_label',
        bins=40,
        element='step',
        common_norm=False,
        ax=axis,
    )
    axis.set_title(f'{column} by target')
    axis.set_xlabel(column)
    axis.legend(title='Target', labels=['Reading', 'Not reading'])
plt.tight_layout()
plt.show()

# Plot 3: correlation heatmap for the complete CNN input.
plt.figure(figsize=(11, 8))
sns.heatmap(motion_df[reading_feature_names].corr(), cmap='coolwarm', center=0)
plt.title('Five-second CNN feature correlations')
plt.tight_layout()
plt.show()

# Plot 4: representative five-second traces for each original behavior class.
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
for axis, original_label, title in zip(
    axes,
    [0, 1, 2],
    ['Normal behavior', 'Looking down', 'Reading'],
):
    source_group = df[df['class_label'] == original_label].sort_values('frame_number').head(75)
    trace = add_motion_features(source_group[feature_cols].to_numpy(dtype='float32'), sampling_fps=15)
    trace_df = pd.DataFrame(trace, columns=reading_feature_names)
    time_seconds = np.arange(len(trace_df)) / 15
    axis.plot(time_seconds, trace_df['yaw'], label='yaw')
    axis.plot(time_seconds, trace_df['gaze_ratio'], label='gaze_ratio')
    axis.plot(time_seconds, trace_df['horizontal_motion'], label='horizontal_motion')
    axis.set_title(f'{title}: representative 5-second trace')
    axis.set_ylabel('Feature value')
    axis.legend(loc='upper right')
axes[-1].set_xlabel('Time (seconds)')
plt.tight_layout()
plt.show()


# EDA for the CNN interview-proctoring model

This notebook reads the extracted feature CSV and checks the actual signal distribution used by the 1D CNN.

- Data source: `data/features.csv`
- Windowing config: `window_size=60`, `stride=10`
- Target: 3-class behavior classification {0: not cheating, 1: looking down, 2: reading}
